# Chapter 4: Model Implementation
## PhishShield — Explainable Pre-Click Phishing URL Detection

This notebook is the executable companion to **Chapter 4: Model Implementation**.

### Deliverables produced
- Trained supervised classification models
- Validation metrics and comparison table
- URL-only champion model
- Full-feature champion model
- Locked test-set metrics
- Decision thresholds
- Precision–Recall and ROC curves
- Confusion matrices
- Calibration diagnostics
- Optional feature-ablation experiment
- Saved model/configuration artifacts under `chapter4_artifacts/`

### Modelling scope
The project is a labelled **binary classification** problem:
- `is_phishing = 1` → phishing
- `is_phishing = 0` → legitimate

Core models:
1. Dummy Classifier
2. Logistic Regression
3. Decision Tree
4. Random Forest
5. Histogram Gradient Boosting
6. XGBoost, if installed
7. Linear SVM with probability calibration
8. Optional MLP

K-Means, DBSCAN, hierarchical clustering, and recommendation methods are not used as core models because they do not match the labelled prediction objective. RNN/CNN/LSTM/Transformer models are better reserved for a future raw-URL sequence experiment.

The test set remains locked until champion selection is complete.

## 4.1 Environment and Reproducibility Configuration

In [ ]:
from pathlib import Path
import json
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "is_phishing"

FAST_MODE = True
MAX_TRAIN_ROWS_FAST = 80_000

RUN_URL_ONLY = True
RUN_FULL_FEATURE = True
RUN_XGBOOST = True
RUN_MLP = False

TARGET_RECALL_FOR_THRESHOLD = 0.90

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("Platform:", platform.platform())
print("FAST_MODE:", FAST_MODE)

> **Final-submission instruction:** Run first with `FAST_MODE=True` to verify the workflow. Before reporting final capstone metrics, set `FAST_MODE=False`, restart the kernel, and run all cells from the top.

## 4.2 Load the Chapter 3 Processed Dataset

In [ ]:
candidate_paths = [
    Path("outputs_step3/phiusiil_step3_processed.csv"),
    Path("phiusiil_step3_processed.csv"),
    Path("../outputs_step3/phiusiil_step3_processed.csv"),
    Path("/mnt/data/outputs_step3/phiusiil_step3_processed.csv"),
    Path("/mnt/data/phiusiil_step3_processed.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find 'phiusiil_step3_processed.csv'. "
        "Run Chapter 3 first or place the processed CSV beside this notebook."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", DATA_PATH.resolve())
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())

In [ ]:
if TARGET not in df.columns:
    if "label" in df.columns:
        print("is_phishing not found; recreating it from label.")
        df[TARGET] = 1 - df["label"].astype(int)
    else:
        raise KeyError(f"Neither '{TARGET}' nor 'label' exists.")

if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError("Target must contain only 0 and 1.")

display(
    df[TARGET]
    .value_counts()
    .rename_axis(TARGET)
    .to_frame("count")
)

## 4.3 Recover the Train / Validation / Test Split

In [ ]:
def normalize_domain_fallback(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if not text:
        return np.nan
    text = text.replace("https://", "").replace("http://", "")
    text = text.split("/")[0]
    text = text.split(":")[0]
    return text.rstrip(".") or np.nan


if "split" not in df.columns:
    print("No split column found. Recreating a domain-grouped 70/15/15 split.")

    from sklearn.model_selection import GroupShuffleSplit

    if "Domain_normalized" not in df.columns:
        if "Domain" in df.columns:
            df["Domain_normalized"] = df["Domain"].map(normalize_domain_fallback)
        elif "URL" in df.columns:
            df["Domain_normalized"] = df["URL"].map(normalize_domain_fallback)
        else:
            raise KeyError("A domain-like field is required to recreate grouped splits.")

    missing_domain = df["Domain_normalized"].isna()
    if missing_domain.any():
        df.loc[missing_domain, "Domain_normalized"] = [
            f"__missing_domain_{i}" for i in df.index[missing_domain]
        ]

    groups = df["Domain_normalized"]

    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=RANDOM_STATE)
    train_idx, temp_idx = next(gss1.split(df, y=df[TARGET], groups=groups))

    temp_local = df.iloc[temp_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=RANDOM_STATE)
    val_rel_idx, test_rel_idx = next(
        gss2.split(
            temp_local,
            y=temp_local[TARGET],
            groups=temp_local["Domain_normalized"]
        )
    )

    df["split"] = None
    df.loc[df.index[train_idx], "split"] = "train"
    df.loc[temp_local.index[val_rel_idx], "split"] = "validation"
    df.loc[temp_local.index[test_rel_idx], "split"] = "test"


train_df = df.loc[df["split"] == "train"].copy()
val_df = df.loc[df["split"] == "validation"].copy()
test_df = df.loc[df["split"] == "test"].copy()

if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("Expected split labels: train, validation, test.")

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "percent": [
        len(train_df) / len(df) * 100,
        len(val_df) / len(df) * 100,
        len(test_df) / len(df) * 100,
    ],
    "phishing_rate": [
        train_df[TARGET].mean(),
        val_df[TARGET].mean(),
        test_df[TARGET].mean(),
    ],
})

display(split_summary.round(4))

In [ ]:
if "Domain_normalized" in df.columns:
    train_domains = set(train_df["Domain_normalized"].dropna())
    val_domains = set(val_df["Domain_normalized"].dropna())
    test_domains = set(test_df["Domain_normalized"].dropna())

    assert train_domains.isdisjoint(val_domains)
    assert train_domains.isdisjoint(test_domains)
    assert val_domains.isdisjoint(test_domains)

    print("Confirmed: no Domain_normalized overlap across splits.")
else:
    print("Domain_normalized unavailable; trusting Chapter 3 split membership.")

## 4.4 Define the Two Modelling Experiments

In [ ]:
published_url_only = [
    "URLLength", "DomainLength", "IsDomainIP", "TLDLength",
    "NoOfSubDomain", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL",
    "NoOfQMarkInURL", "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL",
    "IsHTTPS", "HasObfuscation", "NoOfObfuscatedChar",
    "ObfuscationRatio",
]

full_only_markers = [
    "webpage_complexity", "form_risk", "external_self_ref", "commerce",
    "log1p_LineOfCode", "log1p_LargestLineLength", "log1p_NoOfImage",
    "log1p_NoOfCSS", "log1p_NoOfJS", "log1p_NoOfSelfRef",
    "log1p_NoOfEmptyRef", "log1p_NoOfExternalRef",
    "log1p_NoOfURLRedirect", "log1p_NoOfSelfRedirect",
    "log1p_NoOfPopup", "log1p_NoOfiFrame",
]

engineered_url_only = [
    c for c in df.columns
    if c.startswith("eng_")
    and not any(marker in c for marker in full_only_markers)
    and pd.api.types.is_numeric_dtype(df[c])
]

url_only_features = list(dict.fromkeys([
    c for c in published_url_only + engineered_url_only
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]))

full_numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in {"label", TARGET}
]

leakage_review_features = [
    c for c in [
        "URLSimilarityIndex",
        "CharContinuationRate",
        "TLDLegitimateProb",
        "URLCharProb",
        "DomainTitleMatchScore",
        "URLTitleMatchScore",
    ]
    if c in full_numeric_features
]

if not url_only_features:
    raise ValueError("No URL-only features were found.")

feature_set_summary = pd.DataFrame({
    "Experiment": ["URL-only", "Full-feature"],
    "Purpose": [
        "Safe pre-click deployment candidate",
        "Analytical upper-bound experiment"
    ],
    "Number of numeric predictors": [
        len(url_only_features),
        len(full_numeric_features)
    ]
})

display(feature_set_summary)
print("Ablation candidates:", leakage_review_features)

## 4.5 Computational Sampling for Development Runs

In [ ]:
def make_training_frame(frame):
    """
    Stratified sample that preserves all columns.
    Avoids groupby.apply compatibility issues in recent pandas versions.
    """
    if (not FAST_MODE) or len(frame) <= MAX_TRAIN_ROWS_FAST:
        return frame.copy()

    if TARGET not in frame.columns:
        raise KeyError(f"Target column '{TARGET}' is missing.")

    class_counts = frame[TARGET].value_counts()
    total_rows = len(frame)
    parts = []

    for class_value, class_count in class_counts.items():
        class_frame = frame.loc[frame[TARGET] == class_value].copy()

        n_class = round(
            MAX_TRAIN_ROWS_FAST * class_count / total_rows
        )
        n_class = max(1, min(int(n_class), len(class_frame)))

        parts.append(
            class_frame.sample(
                n=n_class,
                random_state=RANDOM_STATE
            )
        )

    sampled = (
        pd.concat(parts, axis=0)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

    assert TARGET in sampled.columns
    return sampled


fit_train_df = make_training_frame(train_df)

print(f"Original training rows: {len(train_df):,}")
print(f"Rows used for fitting: {len(fit_train_df):,}")
print("Fitting phishing rate:", round(fit_train_df[TARGET].mean(), 4))

display(
    fit_train_df[TARGET]
    .value_counts()
    .rename_axis(TARGET)
    .to_frame("count")
)

## 4.6 Supervised Models

In [ ]:
def scaled_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


def tree_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


def build_model_templates():
    models = {
        "Dummy": tree_pipeline(
            DummyClassifier(
                strategy="prior",
                random_state=RANDOM_STATE
            )
        ),

        "Logistic Regression": scaled_pipeline(
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=3000,
                solver="liblinear",
                random_state=RANDOM_STATE
            )
        ),

        "Decision Tree": tree_pipeline(
            DecisionTreeClassifier(
                max_depth=12,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        ),

        "Random Forest": tree_pipeline(
            RandomForestClassifier(
                n_estimators=250 if FAST_MODE else 500,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE
            )
        ),

        "HistGradientBoosting": tree_pipeline(
            HistGradientBoostingClassifier(
                learning_rate=0.08,
                max_iter=150 if FAST_MODE else 300,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=RANDOM_STATE
            )
        ),

        "Linear SVM": scaled_pipeline(
            CalibratedClassifierCV(
                estimator=LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE
                ),
                method="sigmoid",
                cv=3
            )
        ),
    }

    if RUN_MLP:
        models["MLP"] = scaled_pipeline(
            MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                alpha=1e-4,
                batch_size=512,
                learning_rate_init=1e-3,
                max_iter=60 if FAST_MODE else 150,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=RANDOM_STATE
            )
        )

    if RUN_XGBOOST:
        try:
            from xgboost import XGBClassifier

            models["XGBoost"] = tree_pipeline(
                XGBClassifier(
                    n_estimators=250 if FAST_MODE else 500,
                    max_depth=6,
                    learning_rate=0.08,
                    subsample=0.85,
                    colsample_bytree=0.85,
                    reg_lambda=1.0,
                    reg_alpha=0.0,
                    eval_metric="logloss",
                    n_jobs=-1,
                    random_state=RANDOM_STATE
                )
            )
            print("XGBoost available and added.")

        except Exception as exc:
            print("XGBoost unavailable; continuing without it.")
            print("Reason:", repr(exc))

    return models


model_templates = build_model_templates()
print("Models configured:", list(model_templates))

### Model rationale

| Model | Role |
|---|---|
| Dummy | No-skill baseline |
| Logistic Regression | Interpretable linear baseline |
| Decision Tree | Transparent nonlinear baseline |
| Random Forest | Robust ensemble model |
| Histogram Gradient Boosting | Efficient boosted-tree benchmark |
| XGBoost | Optional high-performance boosting candidate |
| Linear SVM | Margin-based comparator with calibrated probabilities |
| MLP | Optional neural-network extension |

Unsupervised and recommendation methods are not executed because they do not answer the labelled phishing-classification question.

## 4.7 Evaluation Metrics and Threshold Policy

In [ ]:
def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        values = model.predict_proba(X)[:, 1]
        return np.asarray(values), True

    if hasattr(model, "decision_function"):
        values = model.decision_function(X)
        return np.asarray(values).ravel(), False

    return np.asarray(model.predict(X), dtype=float), False


def false_positive_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return fp / (fp + tn) if (fp + tn) else np.nan


def select_threshold_for_target_recall(
    y_true,
    scores,
    minimum_recall=0.90
):
    precision, recall, thresholds = precision_recall_curve(
        y_true, scores
    )

    p = precision[:-1]
    r = recall[:-1]

    eligible = np.where(r >= minimum_recall)[0]

    if len(eligible):
        best = eligible[np.argmax(p[eligible])]
        return float(thresholds[best]), "highest_precision_at_target_recall"

    f1_values = 2 * p * r / np.maximum(p + r, 1e-12)
    best = int(np.nanargmax(f1_values))

    return float(thresholds[best]), "max_f1_fallback"


def calculate_metrics(
    y_true,
    scores,
    threshold,
    probability_output
):
    y_pred = (scores >= threshold).astype(int)

    result = {
        "pr_auc": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positive_rate": false_positive_rate(y_true, y_pred),
        "threshold": float(threshold),
    }

    result["brier_score"] = (
        brier_score_loss(y_true, np.clip(scores, 0, 1))
        if probability_output
        else np.nan
    )

    return result


def latency_ms_per_1000(model, X, max_rows=5000):
    sample = X.iloc[:min(max_rows, len(X))]

    if len(sample) == 0:
        return np.nan

    start = time.perf_counter()
    _ = predict_scores(model, sample)
    elapsed = time.perf_counter() - start

    return elapsed / len(sample) * 1_000_000

**Primary metric:** PR-AUC / Average Precision.

**Supporting metrics:** recall, precision, F1, ROC-AUC, accuracy, false-positive rate, Brier score, training time, and inference latency.

The classification threshold is selected from validation data only.

## 4.8 Train Models and Compare Validation Performance

In [ ]:
ARTIFACT_ROOT = Path("chapter4_artifacts")
MODELS_DIR = ARTIFACT_ROOT / "models"
FIGURES_DIR = ARTIFACT_ROOT / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

experiments = {}

if RUN_URL_ONLY:
    experiments["URL-only"] = url_only_features

if RUN_FULL_FEATURE:
    experiments["Full-feature"] = full_numeric_features

fitted_models = {}
thresholds = {}
validation_records = []

for experiment_name, features in experiments.items():

    print("\n" + "=" * 90)
    print(f"EXPERIMENT: {experiment_name}")
    print(f"Predictors: {len(features)}")
    print("=" * 90)

    X_train = fit_train_df[features].replace([np.inf, -np.inf], np.nan)
    y_train = fit_train_df[TARGET].astype(int)

    X_val = val_df[features].replace([np.inf, -np.inf], np.nan)
    y_val = val_df[TARGET].astype(int)

    for model_name, template in model_templates.items():

        key = f"{experiment_name}__{model_name}"
        print(f"\nTraining {key}")

        model = clone(template)

        start = time.perf_counter()
        model.fit(X_train, y_train)
        training_seconds = time.perf_counter() - start

        val_scores, probability_output = predict_scores(model, X_val)

        threshold, threshold_rule = select_threshold_for_target_recall(
            y_val,
            val_scores,
            minimum_recall=TARGET_RECALL_FOR_THRESHOLD
        )

        metrics = calculate_metrics(
            y_val,
            val_scores,
            threshold,
            probability_output
        )

        latency = latency_ms_per_1000(model, X_val)

        validation_records.append({
            "experiment": experiment_name,
            "model": model_name,
            "n_features": len(features),
            "training_rows": len(X_train),
            "training_seconds": training_seconds,
            "inference_ms_per_1000": latency,
            "probability_output": probability_output,
            "threshold_rule": threshold_rule,
            **metrics,
        })

        fitted_models[key] = model
        thresholds[key] = threshold

        safe_filename = (
            key.lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

        joblib.dump(
            model,
            MODELS_DIR / f"{safe_filename}.joblib"
        )

        print(
            f"PR-AUC={metrics['pr_auc']:.6f} | "
            f"Recall={metrics['recall']:.6f} | "
            f"Precision={metrics['precision']:.6f} | "
            f"F1={metrics['f1']:.6f}"
        )


validation_results = (
    pd.DataFrame(validation_records)
    .sort_values(
        ["experiment", "pr_auc", "f1"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

display(validation_results.round(6))

## 4.9 Model Comparison

In [ ]:
comparison_columns = [
    "experiment",
    "model",
    "n_features",
    "pr_auc",
    "recall",
    "precision",
    "f1",
    "roc_auc",
    "accuracy",
    "false_positive_rate",
    "brier_score",
    "threshold",
    "training_seconds",
    "inference_ms_per_1000",
]

model_comparison = validation_results[comparison_columns].copy()

display(model_comparison.round(6))

model_comparison.to_csv(
    ARTIFACT_ROOT / "validation_model_comparison.csv",
    index=False
)

In [ ]:
plot_df = validation_results.sort_values("pr_auc", ascending=True)
labels = plot_df["experiment"] + " | " + plot_df["model"]

fig, ax = plt.subplots(
    figsize=(10, max(6, 0.45 * len(plot_df)))
)

ax.barh(labels, plot_df["pr_auc"])
ax.set_xlabel("Validation PR-AUC / Average Precision")
ax.set_title("Validation Model Comparison — Primary Metric")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "validation_pr_auc_comparison.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()

## 4.10 Precision–Recall Curves

In [ ]:
for experiment_name, features in experiments.items():

    X_val = val_df[features].replace([np.inf, -np.inf], np.nan)
    y_val = val_df[TARGET].astype(int)

    fig, ax = plt.subplots(figsize=(8, 6))

    for model_name in model_templates:
        key = f"{experiment_name}__{model_name}"

        if key not in fitted_models:
            continue

        scores, _ = predict_scores(fitted_models[key], X_val)

        precision, recall, _ = precision_recall_curve(y_val, scores)
        ap = average_precision_score(y_val, scores)

        ax.plot(
            recall,
            precision,
            label=f"{model_name} (AP={ap:.4f})"
        )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(
        f"Validation Precision–Recall Curves — {experiment_name}"
    )
    ax.legend(fontsize=8)
    plt.tight_layout()

    safe = experiment_name.lower().replace("-", "_").replace(" ", "_")

    plt.savefig(
        FIGURES_DIR / f"{safe}_validation_pr_curves.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.show()

## 4.11 ROC Curves

In [ ]:
for experiment_name, features in experiments.items():

    X_val = val_df[features].replace([np.inf, -np.inf], np.nan)
    y_val = val_df[TARGET].astype(int)

    fig, ax = plt.subplots(figsize=(8, 6))

    for model_name in model_templates:
        key = f"{experiment_name}__{model_name}"

        if key not in fitted_models:
            continue

        scores, _ = predict_scores(fitted_models[key], X_val)

        fpr, tpr, _ = roc_curve(y_val, scores)
        auc = roc_auc_score(y_val, scores)

        ax.plot(
            fpr,
            tpr,
            label=f"{model_name} (AUC={auc:.4f})"
        )

    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Validation ROC Curves — {experiment_name}")
    ax.legend(fontsize=8)
    plt.tight_layout()

    safe = experiment_name.lower().replace("-", "_").replace(" ", "_")

    plt.savefig(
        FIGURES_DIR / f"{safe}_validation_roc_curves.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.show()

## 4.12 Champion Selection

In [ ]:
champions = {}

for experiment_name in experiments:

    candidates = validation_results.loc[
        validation_results["experiment"] == experiment_name
    ].copy()

    winner = candidates.sort_values(
        ["pr_auc", "f1"],
        ascending=False
    ).iloc[0]

    model_name = winner["model"]
    key = f"{experiment_name}__{model_name}"

    champions[experiment_name] = {
        "key": key,
        "model": model_name,
        "threshold": float(thresholds[key]),
        "validation_pr_auc": float(winner["pr_auc"]),
        "validation_recall": float(winner["recall"]),
        "validation_precision": float(winner["precision"]),
        "validation_f1": float(winner["f1"]),
        "validation_false_positive_rate": float(
            winner["false_positive_rate"]
        ),
    }

champion_table = pd.DataFrame(champions).T
display(champion_table)

Two champions are retained:
- **URL-only champion:** preferred safe pre-click deployment candidate.
- **Full-feature champion:** analytical upper bound using webpage-derived information.

## 4.13 Locked Test-Set Evaluation

In [ ]:
test_records = []

for experiment_name, champion in champions.items():

    features = experiments[experiment_name]
    model = fitted_models[champion["key"]]
    threshold = champion["threshold"]

    X_test = test_df[features].replace([np.inf, -np.inf], np.nan)
    y_test = test_df[TARGET].astype(int)

    scores, probability_output = predict_scores(model, X_test)

    metrics = calculate_metrics(
        y_test,
        scores,
        threshold,
        probability_output
    )

    latency = latency_ms_per_1000(model, X_test)

    test_records.append({
        "experiment": experiment_name,
        "model": champion["model"],
        "n_features": len(features),
        "inference_ms_per_1000": latency,
        **metrics,
    })

    y_pred = (scores >= threshold).astype(int)

    print("\n" + "=" * 90)
    print(f"LOCKED TEST: {experiment_name} | {champion['model']}")
    print("=" * 90)

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Legitimate", "Phishing"],
            digits=6
        )
    )

test_results = pd.DataFrame(test_records)

display(test_results.round(6))

test_results.to_csv(
    ARTIFACT_ROOT / "champion_test_metrics.csv",
    index=False
)

## 4.14 Champion Confusion Matrices

In [ ]:
for experiment_name, champion in champions.items():

    features = experiments[experiment_name]
    model = fitted_models[champion["key"]]
    threshold = champion["threshold"]

    X_test = test_df[features].replace([np.inf, -np.inf], np.nan)
    y_test = test_df[TARGET].astype(int)

    scores, _ = predict_scores(model, X_test)
    y_pred = (scores >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.imshow(cm)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Legitimate", "Phishing"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Legitimate", "Phishing"])

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_title(
        f"Test Confusion Matrix\n"
        f"{experiment_name} — {champion['model']}"
    )

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i, f"{cm[i, j]:,}",
                ha="center", va="center"
            )

    plt.tight_layout()

    safe = experiment_name.lower().replace("-", "_").replace(" ", "_")

    plt.savefig(
        FIGURES_DIR / f"{safe}_champion_test_confusion_matrix.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.show()

## 4.15 Probability Calibration Diagnostics

In [ ]:
for experiment_name, champion in champions.items():

    features = experiments[experiment_name]
    model = fitted_models[champion["key"]]

    X_test = test_df[features].replace([np.inf, -np.inf], np.nan)
    y_test = test_df[TARGET].astype(int)

    scores, probability_output = predict_scores(model, X_test)

    if not probability_output:
        print(
            f"{experiment_name} / {champion['model']}: "
            "calibration skipped."
        )
        continue

    scores = np.clip(scores, 0, 1)

    observed, predicted = calibration_curve(
        y_test,
        scores,
        n_bins=10,
        strategy="quantile"
    )

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    ax.plot(
        predicted,
        observed,
        marker="o",
        label=champion["model"]
    )

    ax.set_xlabel("Mean predicted phishing probability")
    ax.set_ylabel("Observed phishing frequency")
    ax.set_title(f"Test Calibration — {experiment_name}")
    ax.legend()

    plt.tight_layout()

    safe = experiment_name.lower().replace("-", "_").replace(" ", "_")

    plt.savefig(
        FIGURES_DIR / f"{safe}_champion_test_calibration.png",
        dpi=160,
        bbox_inches="tight"
    )
    plt.show()

## 4.16 Full-Feature Ablation Experiment

In [ ]:
RUN_FULL_FEATURE_ABLATION = True
ablation_result = None

if (
    RUN_FULL_FEATURE_ABLATION
    and "Full-feature" in champions
    and leakage_review_features
):

    ablation_features = [
        c for c in full_numeric_features
        if c not in leakage_review_features
    ]

    champion_model_name = champions["Full-feature"]["model"]
    ablation_model = clone(model_templates[champion_model_name])

    X_train = fit_train_df[ablation_features].replace(
        [np.inf, -np.inf], np.nan
    )
    y_train = fit_train_df[TARGET].astype(int)

    X_val = val_df[ablation_features].replace(
        [np.inf, -np.inf], np.nan
    )
    y_val = val_df[TARGET].astype(int)

    print("Removed:", leakage_review_features)
    print("Remaining predictors:", len(ablation_features))

    ablation_model.fit(X_train, y_train)

    scores, probability_output = predict_scores(
        ablation_model,
        X_val
    )

    threshold, threshold_rule = select_threshold_for_target_recall(
        y_val,
        scores,
        TARGET_RECALL_FOR_THRESHOLD
    )

    metrics = calculate_metrics(
        y_val,
        scores,
        threshold,
        probability_output
    )

    ablation_result = {
        "model": champion_model_name,
        "removed_features": leakage_review_features,
        "remaining_features": len(ablation_features),
        "threshold_rule": threshold_rule,
        **metrics,
    }

    display(pd.Series(ablation_result, name="value").to_frame())

else:
    print("Ablation skipped.")

The ablation test measures whether the full-feature result depends heavily on precomputed similarity/probability variables. It does not by itself prove or disprove leakage.

## 4.17 Optional Grouped Hyperparameter Tuning

In [ ]:
RUN_HYPERPARAMETER_TUNING = False

if RUN_HYPERPARAMETER_TUNING:

    from sklearn.model_selection import GroupKFold, RandomizedSearchCV

    if "Domain_normalized" not in train_df.columns:
        raise KeyError(
            "Domain_normalized is required for grouped cross-validation."
        )

    X_tune = train_df[url_only_features].replace(
        [np.inf, -np.inf], np.nan
    )
    y_tune = train_df[TARGET].astype(int)
    groups_tune = train_df["Domain_normalized"]

    rf_template = tree_pipeline(
        RandomForestClassifier(
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )

    parameter_space = {
        "model__n_estimators": [250, 400, 600],
        "model__max_depth": [None, 12, 20, 30],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": ["sqrt", "log2", 0.5],
    }

    grouped_cv = GroupKFold(n_splits=3)

    search = RandomizedSearchCV(
        estimator=rf_template,
        param_distributions=parameter_space,
        n_iter=12,
        scoring="average_precision",
        cv=grouped_cv,
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE
    )

    search.fit(
        X_tune,
        y_tune,
        groups=groups_tune
    )

    print("Best grouped-CV PR-AUC:", search.best_score_)
    print("Best parameters:", search.best_params_)

    joblib.dump(
        search.best_estimator_,
        MODELS_DIR / "url_only_random_forest_tuned.joblib"
    )

## 4.18 Reproducibility: Save Configurations and Artifacts

In [ ]:
feature_schema = {
    "url_only_features": url_only_features,
    "full_numeric_features": full_numeric_features,
    "leakage_review_features": leakage_review_features,
}

with open(
    ARTIFACT_ROOT / "feature_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(feature_schema, f, indent=2)


with open(
    ARTIFACT_ROOT / "thresholds.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {k: float(v) for k, v in thresholds.items()},
        f,
        indent=2
    )


with open(
    ARTIFACT_ROOT / "champions.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(champions, f, indent=2)


if ablation_result is not None:
    with open(
        ARTIFACT_ROOT / "full_feature_ablation.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            ablation_result,
            f,
            indent=2,
            default=str
        )


config = {
    "random_state": RANDOM_STATE,
    "fast_mode": FAST_MODE,
    "maximum_fast_training_rows": MAX_TRAIN_ROWS_FAST,
    "target": TARGET,
    "positive_class": "phishing",
    "primary_metric": "PR-AUC / Average Precision",
    "minimum_recall_threshold_policy": TARGET_RECALL_FOR_THRESHOLD,
    "run_url_only": RUN_URL_ONLY,
    "run_full_feature": RUN_FULL_FEATURE,
    "run_xgboost": RUN_XGBOOST,
    "run_mlp": RUN_MLP,
    "split_strategy": "Chapter 3 domain-grouped train/validation/test split",
    "test_policy": "Locked until validation-based champion and threshold selection",
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "platform": platform.platform(),
}

with open(
    ARTIFACT_ROOT / "config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)


for experiment_name, champion in champions.items():

    model = fitted_models[champion["key"]]

    stable_name = (
        "url_only_champion.joblib"
        if experiment_name == "URL-only"
        else "full_feature_champion.joblib"
    )

    joblib.dump(
        model,
        MODELS_DIR / stable_name
    )

print("Artifacts saved to:", ARTIFACT_ROOT.resolve())

In [ ]:
artifact_inventory = [
    str(p.relative_to(ARTIFACT_ROOT))
    for p in sorted(ARTIFACT_ROOT.rglob("*"))
    if p.is_file()
]

display(
    pd.DataFrame({
        "Saved artifact": artifact_inventory
    })
)

## 4.19 Automatically Generated Report Summary

In [ ]:
print("=" * 90)
print("CHAPTER 4 — REPORT-READY SUMMARY")
print("=" * 90)

print(
    "\nTraining mode:",
    "FAST development run" if FAST_MODE else "FULL training run"
)

print(f"Training observations used: {len(fit_train_df):,}")

for experiment_name, champion in champions.items():

    test_row = test_results.loc[
        test_results["experiment"] == experiment_name
    ].iloc[0]

    print("\n" + "-" * 90)
    print(f"{experiment_name} champion: {champion['model']}")
    print(
        f"Validation PR-AUC: "
        f"{champion['validation_pr_auc']:.6f}"
    )
    print(
        f"Locked-test PR-AUC: "
        f"{test_row['pr_auc']:.6f}"
    )
    print(
        f"Locked-test recall: "
        f"{test_row['recall']:.6f}"
    )
    print(
        f"Locked-test precision: "
        f"{test_row['precision']:.6f}"
    )
    print(
        f"Locked-test F1: "
        f"{test_row['f1']:.6f}"
    )
    print(
        f"Locked-test false-positive rate: "
        f"{test_row['false_positive_rate']:.6f}"
    )

print("\n" + "=" * 90)

if FAST_MODE:
    print(
        "IMPORTANT: Development-run results only. "
        "Set FAST_MODE=False and rerun before using final capstone metrics."
    )
else:
    print(
        "Full-training run complete. "
        "Review these results before inserting them in the final report."
    )

## 4.20 Deliverable Checklist

After the final run, confirm that `chapter4_artifacts/` contains:

### Trained models
- Individual fitted model pipelines
- `models/url_only_champion.joblib`
- `models/full_feature_champion.joblib`

### Metrics and comparisons
- `validation_model_comparison.csv`
- `champion_test_metrics.csv`
- `thresholds.json`
- `champions.json`

### Reproducibility
- `config.json`
- `feature_schema.json`
- optional `full_feature_ablation.json`

### Figures
- Validation PR-AUC comparison
- Precision–Recall curves
- ROC curves
- Champion confusion matrices
- Champion calibration plots

**Deliverable satisfied:** trained models, metrics, comparison between models, selected champions, and reproducible model artifacts.